# 02. Phân Tích Các Tác Tử Cổ Điển (Level 1 – Level 4)

**Mục tiêu:**
1. Kiểm chứng hàm lượng giá tĩnh (`evaluate_board`: Material + PST Michniewski + King Safety + Mobility + Cấu trúc Tốt + Xe cột mở + Cặp Tượng).
2. Phân tích hành vi ra quyết định và tốc độ suy luận (`NPS`/`SPS`, Latency) của **Level 1 (Random)**, **Level 2 (Heuristic 6 Tiers + SEE)**, **Level 3 (Alpha-Beta + PVS + Aspiration + TT + Quiescence/SEE)** và **Level 4 (Vanilla MCTS UCT)**.
3. **Thí nghiệm đối chứng Level 4:** So sánh trực tiếp **`rollout_mode="see_tactical"`** (rollout có luật SEE) vs **`rollout_mode="uniform"`** (rollout ngẫu nhiên hoàn toàn) về tỷ lệ thắng, độ trễ và tốc độ `SPS`.

In [ ]:
from functools import partial
import json
from pathlib import Path
import sys

import chess
from IPython.display import display
import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from chess_ai.agents.level4_mcts import Level4MCTSAgent
from chess_ai.agents.registry import create_agent
from chess_ai.arena.tournament import run_paired_matchup
from chess_ai.core.chess_env import ChessEnv
from chess_ai.core.evaluation import (
    bishop_pair_score,
    evaluate_board,
    pawn_structure_score,
    rook_file_score,
)
from chess_ai.utils.epd import load_epd_file

## 1. Phân Tích Hàm Lượng Giá Tĩnh & Các Thành Phần Vị Trí

In [ ]:
sample_fens = {
    "Khai cuộc chuẩn": chess.STARTING_FEN,
    "Trung cuộc cân bằng (Italian)": "r1bqkbnr/pppp1ppp/2n5/4p3/2B1P3/5N2/PPPP1PPP/RNBQK2R w KQkq - 4 4",
    "Tàn cuộc Tốt thông e6 + Cặp Tượng": "4k3/p7/p3P3/8/8/8/8/2B1K1B1 w - - 0 1",
}

eval_rows = []
for label, fen in sample_fens.items():
    b = chess.Board(fen)
    eval_rows.append({
        "Thế cờ": label,
        "Evaluate (cp)": round(evaluate_board(b), 2),
        "Pawn Struct Δ": pawn_structure_score(b, chess.WHITE) - pawn_structure_score(b, chess.BLACK),
        "Rook File Δ": rook_file_score(b, chess.WHITE) - rook_file_score(b, chess.BLACK),
        "Bishop Pair Δ": bishop_pair_score(b, chess.WHITE) - bishop_pair_score(b, chess.BLACK),
    })

display(pd.DataFrame(eval_rows))

## 2. So Sánh Thời Gian & Số Nút Duyệt Giữa Level 1 – Level 4

In [ ]:
probe_fen = "r1bqkbnr/pppp1ppp/2n5/4p3/2B1P3/5N2/PPPP1PPP/RNBQK2R w KQkq - 4 4"
bench_rows = []
for aid in ("level1", "level2", "level3", "level4"):
    env = ChessEnv(fen=probe_fen)
    ag = create_agent(aid)
    m = ag.select_move(env, max_workload=400)
    st = ag.last_search_stats
    t_sec = max(1e-6, float(st.get("time_sec", 0.0)))
    nodes = float(st.get("nodes", 0.0))
    sims = float(st.get("simulations", 0.0))
    bench_rows.append({
        "Agent": ag.name,
        "Nước chọn": env.board.san(m),
        "Nodes": int(nodes),
        "Simulations": int(sims),
        "Time (ms)": round(t_sec * 1000.0, 2),
        "Throughput (NPS/SPS)": round((sims if sims > 0 else nodes) / t_sec, 1),
    })

display(pd.DataFrame(bench_rows))

## 3. Thí Nghiệm Đối Chứng Level 4: `rollout_mode="see_tactical"` vs `rollout_mode="uniform"`

Chạy đối đầu ghép cặp đối xứng trên bộ khai cuộc cân bằng `resources/openings_50.epd` giữa:
- **L4 (`see_tactical`):** Rollout ưu tiên 70% nước bắt quân không lỗ (`see_capture >= 0`) hoặc phong Hậu.
- **L4 (`uniform`):** Rollout ngẫu nhiên đồng đều hoàn toàn.

Đo tỷ lệ điểm $S$, $W/D/L$, khoảng tin cậy Wilson/Bootstrap 95%, độ trễ trung bình và tốc độ mô phỏng (`SPS`).

In [ ]:
NUM_OPENINGS = 15  # 15 khai cuộc x 2 màu = 30 ván (tăng lên 25 hoặc 50 khi chạy chính thức)
N_SIM = 200
SEEDS = [42, 101, 2024, 777, 12]

openings = load_epd_file(PROJECT_ROOT / "resources" / "openings_50.epd")[:NUM_OPENINGS]

def make_l4_see() -> Level4MCTSAgent:
    ag = Level4MCTSAgent(simulations=N_SIM, rollout_mode="see_tactical")
    ag.name = "Level4_UCT_SEE_Tactical"
    return ag

def make_l4_uniform() -> Level4MCTSAgent:
    ag = Level4MCTSAgent(simulations=N_SIM, rollout_mode="uniform")
    ag.name = "Level4_UCT_Uniform"
    return ag

res_rollout = run_paired_matchup(
    agent_a=make_l4_see(),
    agent_b=make_l4_uniform(),
    openings=openings,
    seeds=SEEDS,
    budget_mode="workload_control",
    workload_a=N_SIM,
    workload_b=N_SIM,
    max_plies=200,
    bootstrap_resamples=2000,
)

# Lưu kết quả để tái sử dụng cho báo cáo
out_json = PROJECT_ROOT / "results" / "ablation2" / "l4_rollout_ablation.json"
out_json.parent.mkdir(parents=True, exist_ok=True)
out_json.write_text(json.dumps(res_rollout, ensure_ascii=False, indent=2), encoding="utf-8")

sps_see = N_SIM / max(1e-6, res_rollout["agent_a_avg_latency_ms"] / 1000.0)
sps_uni = N_SIM / max(1e-6, res_rollout["agent_b_avg_latency_ms"] / 1000.0)

df_rollout = pd.DataFrame([
    {
        "Biến thể Level 4": "see_tactical (Rollout có luật SEE)",
        "Ván": res_rollout["games"],
        "W / D / L": f"{res_rollout['wins']} / {res_rollout['draws']} / {res_rollout['losses']}",
        "Điểm S": res_rollout["score"],
        "Wilson 95% CI": str(res_rollout["wilson_ci_95"]),
        "Avg ms / nước": res_rollout["agent_a_avg_latency_ms"],
        "SPS (sims/s)": round(sps_see, 1),
    },
    {
        "Biến thể Level 4": "uniform (Rollout ngẫu nhiên thuần)",
        "Ván": res_rollout["games"],
        "W / D / L": f"{res_rollout['losses']} / {res_rollout['draws']} / {res_rollout['wins']}",
        "Điểm S": round(1.0 - res_rollout["score"], 4),
        "Wilson 95% CI": "—",
        "Avg ms / nước": res_rollout["agent_b_avg_latency_ms"],
        "SPS (sims/s)": round(sps_uni, 1),
    },
])
display(df_rollout)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.5, 3.8), dpi=130)
labels = ["see_tactical", "uniform"]
scores = [res_rollout["score"] * 100.0, (1.0 - res_rollout["score"]) * 100.0]
spss = [sps_see, sps_uni]

ax1.bar(labels, scores, color=["#16A34A", "#64748B"], width=0.5)
ax1.set_ylim(0, 105)
ax1.set_ylabel("Tỷ lệ điểm S (%)")
ax1.set_title(f"Sức cờ đối đầu (N_sim={N_SIM}, {res_rollout['games']} ván)")
for i, v in enumerate(scores):
    ax1.text(i, v + 2.0, f"{v:.1f}%", ha="center", fontweight="bold")

ax2.bar(labels, spss, color=["#3B82F6", "#F59E0B"], width=0.5)
ax2.set_ylabel("Simulations / giây (SPS)")
ax2.set_title("Đánh đổi tốc độ mô phỏng (SPS)")
for i, v in enumerate(spss):
    ax2.text(i, v + 5.0, f"{v:.1f}", ha="center", fontweight="bold")

fig.tight_layout()
plt.show()